In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio, create_static_tool_filter
from IPython.display import display, Markdown
import os
from pathlib import Path
from agents.extensions.models.litellm_model import LitellmModel
import json

load_dotenv(override=True)
print(f"OpenRouter API Key found: {bool(os.environ.get('OPENROUTER_API_KEY'))}")

In [ ]:
TAVILY_API_KEY = os.getenv('TAVILY_API_KEY')
print(f"Tavily API Key found: {bool(os.environ.get('TAVILY_API_KEY'))}")

In [ ]:
import json
json.dumps({"include_images": False,"include_answer": True,  "search_depth": "advanced", "max_results": 10})

In [ ]:


# params = {
#       "command": "npx",
#       "args": [
#         "-y",
#         "mcp-remote",
#         f"https://mcp.tavily.com/mcp/?tavilyApiKey={TAVILY_API_KEY}"
#       ],
#       "env": {
#         "DEFAULT_PARAMETERS": json.dumps({"include_images": False,"include_answer": True,  "search_depth": "advanced", "max_results": 10})
#       }
# }

# tool_filter=create_static_tool_filter(blocked_tool_names=["tavily_research","tavily_map","tavily_crawl","tavily_skill"])

In [ ]:
## http version
from agents.mcp import MCPServerStdio, MCPServerStreamableHttp, create_static_tool_filter
params = {
    "url": f"https://mcp.tavily.com/mcp/?tavilyApiKey={TAVILY_API_KEY}",
    "headers": {
        "DEFAULT_PARAMETERS": json.dumps({
            "include_images": False,
            "include_answer": "advanced",
            "max_results": 5,
            "search_depth": "advanced",
            "auto_parameters": True
        })
    }
}

tool_filter=create_static_tool_filter(blocked_tool_names=["tavily_research","tavily_map","tavily_crawl","tavily_skill"])

In [ ]:
root_dir = Path(os.getcwd()).parent
abs_tmp_dir = str((root_dir / "tmp").resolve())

mcp_server_python = MCPServerStdio(
    name="Sandboxed Workspace",
    params={ 
        "command": "docker", 
        "args": [ 
            "run", "-i", "--rm", 
            "--security-opt", "no-new-privileges",
            "--cap-drop", "ALL",
            "--init",
            "--memory", "512m",
            "--cpus", "0.5",
            "-v", f"{abs_tmp_dir}:/workspace", 
            "ghcr.io/hrrodan/agent-workspace-mcp:latest" 
        ] 
    }, 
    client_session_timeout_seconds=60.0 
)

In [ ]:
abs_tmp_dir

In [ ]:
instructions = "You have tools available, use them if necessary."
request = "Check the docs from functools online and write a small example script in your workspace. Use the parameters include_answer: True, search_depth: advanced"
model = "openrouter/google/gemini-3-flash-preview"

In [ ]:
async with MCPServerStreamableHttp(params=params, tool_filter=tool_filter, client_session_timeout_seconds=30) as mcp_server:
    async with mcp_server_python as mcp_server_python:
        agent = Agent(name="tool_manager", instructions=instructions, model=LitellmModel(model), mcp_servers=[mcp_server, mcp_server_python])
        with trace("tool_manager"):
            result = await Runner.run(agent, request, max_turns=30)
        display(Markdown(result.final_output))
